In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path.home() / "Desktop" / "FlowCast"
DATA_PATH = PROJECT_DIR / "data" / "processed" / "flowcast_features.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

print("\nAccident count distribution:")
print(df["accident_count"].value_counts(dropna=False).sort_index())

Dataset shape: (176701, 46)

Available columns:
['road_id', 'road_name', 'latitude', 'longitude', 'weather_station_id', 'date', 'time', 'traffic_volume', 'vehicle_count', 'vehicle_type_dist', 'avg_speed', 'occupancy', 'congestion_level', 'travel_time', 'accident_count', 'signal_timing', 'road_capacity', 'timestamp', 'capacity_30min', 'volume_capacity_ratio', 'date_weather', 'time_weather', 'weather_condition', 'temperature', 'rainfall', 'visibility', 'public_holiday', 'holiday_name', 'event_flag', 'event_name', 'roadwork_flag', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'is_morning_peak', 'is_evening_peak', 'traffic_volume_lag_1', 'traffic_volume_lag_2', 'traffic_volume_lag_4', 'avg_speed_lag_1', 'avg_speed_lag_2', 'traffic_volume_rolling_mean_4', 'avg_speed_rolling_mean_4']

Accident count distribution:
accident_count
0    175049
1      1637
2        15
Name: count, dtype: int64


In [3]:
# Create a binary target:
# 0 = No accident recorded
# 1 = One or more accidents recorded

df["accident_risk"] = (
    pd.to_numeric(df["accident_count"], errors="coerce")
    .fillna(0)
    .gt(0)
    .astype(int)
)

print("Accident-risk target created successfully!")
print("\nTarget distribution:")
print(df["accident_risk"].value_counts())

print("\nTarget percentages:")
print(
    (df["accident_risk"].value_counts(normalize=True) * 100)
    .round(2)
)

Accident-risk target created successfully!

Target distribution:
accident_risk
0    175049
1      1652
Name: count, dtype: int64

Target percentages:
accident_risk
0    99.07
1     0.93
Name: proportion, dtype: float64


In [4]:
 import numpy as np
from sklearn.impute import SimpleImputer

# Convert timestamps and sort chronologically
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Select numeric features without the target or accident outcome
exclude_cols = [
    "accident_risk",
    "accident_count",
    "timestamp",
    "congestion_level"
]

feature_cols = [
    col for col in df.select_dtypes(include=np.number).columns
    if col not in exclude_cols
]

X = df[feature_cols].replace([np.inf, -np.inf], np.nan)
y = df["accident_risk"].astype(int)

# Use earlier records for training and later records for testing
split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

# Fill missing values using training data only
imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_cols
)
X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_cols
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features:", len(feature_cols))
print("Accidents in training:", int(y_train.sum()))
print("Accidents in testing:", int(y_test.sum()))
print("Feature names:", feature_cols)

Training rows: 141360
Testing rows: 35341
Features: 32
Accidents in training: 1308
Accidents in testing: 344
Feature names: ['latitude', 'longitude', 'traffic_volume', 'vehicle_count', 'avg_speed', 'occupancy', 'travel_time', 'signal_timing', 'road_capacity', 'capacity_30min', 'volume_capacity_ratio', 'temperature', 'rainfall', 'visibility', 'public_holiday', 'event_flag', 'roadwork_flag', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'is_morning_peak', 'is_evening_peak', 'traffic_volume_lag_1', 'traffic_volume_lag_2', 'traffic_volume_lag_4', 'avg_speed_lag_1', 'avg_speed_lag_2', 'traffic_volume_rolling_mean_4', 'avg_speed_rolling_mean_4']


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    average_precision_score
)
import joblib
import json

# Train a class-balanced model
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Predict accident probabilities
y_probability = model.predict_proba(X_test)[:, 1]

# Use a threshold of 0.5 initially
y_pred = (y_probability >= 0.5).astype(int)

print("ROC-AUC:", round(roc_auc_score(y_test, y_probability), 4))
print("Average Precision:", round(average_precision_score(y_test, y_probability), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

ROC-AUC: 0.5722
Average Precision: 0.0158

Classification report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.98     34997
           1       0.03      0.10      0.04       344

    accuracy                           0.95     35341
   macro avg       0.51      0.53      0.51     35341
weighted avg       0.98      0.95      0.97     35341


Confusion matrix:
[[33652  1345]
 [  309    35]]


In [6]:
print("Accident-risk target definition:")
print("Positive cases:", int(df["accident_risk"].sum()))

print("\nFeatures used by the model:")
for col in feature_cols:
    print(col)

print("\nAccident count by risk label:")
if "accident_count" in df.columns:
    print(
        df.groupby("accident_risk")["accident_count"]
        .agg(["count", "mean", "max"])
    )

Accident-risk target definition:
Positive cases: 1652

Features used by the model:
latitude
longitude
traffic_volume
vehicle_count
avg_speed
occupancy
travel_time
signal_timing
road_capacity
capacity_30min
volume_capacity_ratio
temperature
rainfall
visibility
public_holiday
event_flag
roadwork_flag
hour
minute
day_of_week
day_of_month
month
is_weekend
is_morning_peak
is_evening_peak
traffic_volume_lag_1
traffic_volume_lag_2
traffic_volume_lag_4
avg_speed_lag_1
avg_speed_lag_2
traffic_volume_rolling_mean_4
avg_speed_rolling_mean_4

Accident count by risk label:
                count     mean  max
accident_risk                      
0              175049  0.00000    0
1                1652  1.00908    2


In [7]:
print("Accident count values:")
print(df["accident_count"].value_counts(dropna=False).sort_index())

print("\nAccident-risk target rule:")
print("Please check the earlier cell where accident_risk was created.")

Accident count values:
accident_count
0    175049
1      1637
2        15
Name: count, dtype: int64

Accident-risk target rule:
Please check the earlier cell where accident_risk was created.


In [8]:
print("Rows:", len(df))
print("Timestamp range:")
print(df["timestamp"].min(), "to", df["timestamp"].max())

print("\nAccident counts:")
print(df["accident_count"].value_counts(dropna=False).sort_index())

print("\nSample records:")
print(
    df[["timestamp", "road_id", "accident_count", "accident_risk"]]
    .head(10)
)

Rows: 176701
Timestamp range:
2025-01-01 00:00:00 to 2025-05-31 23:30:00

Accident counts:
accident_count
0    175049
1      1637
2        15
Name: count, dtype: int64

Sample records:
   timestamp road_id  accident_count  accident_risk
0 2025-01-01  NL-001               0              0
1 2025-01-01  NL-012               0              0
2 2025-01-01  NL-015               0              0
3 2025-01-01  NL-013               0              0
4 2025-01-01  NL-014               0              0
5 2025-01-01  NL-003               0              0
6 2025-01-01  NL-017               0              0
7 2025-01-01  NL-010               0              0
8 2025-01-01  NL-022               0              0
9 2025-01-01  NL-005               0              0


In [9]:
print("Timestamp sorted:", df["timestamp"].is_monotonic_increasing)

print("\nAccidents by road:")
print(
    df.groupby("road_id")["accident_count"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

print("\nFirst accident records:")
print(
    df.loc[
        df["accident_count"] > 0,
        ["timestamp", "road_id", "accident_count"]
    ].head(10).to_string(index=False)
)

Timestamp sorted: True

Accidents by road:
road_id
NL-006    86
NL-016    81
NL-017    80
NL-025    80
NL-008    76
NL-024    76
NL-015    75
NL-018    73
NL-013    71
NL-005    69
Name: accident_count, dtype: int64

First accident records:
          timestamp road_id  accident_count
2025-01-01 02:30:00  NL-023               1
2025-01-01 08:30:00  NL-008               1
2025-01-01 09:00:00  NL-021               1
2025-01-01 13:30:00  NL-006               1
2025-01-01 21:30:00  NL-025               1
2025-01-02 03:30:00  NL-019               1
2025-01-02 05:00:00  NL-024               1
2025-01-02 06:00:00  NL-012               1
2025-01-02 06:30:00  NL-009               1
2025-01-02 07:00:00  NL-002               1


In [10]:
# Sort records by road and time
df = df.sort_values(
    ["road_id", "timestamp"]
).reset_index(drop=True)

# Check whether every road has regular 30-minute intervals
time_gaps = (
    df.groupby("road_id")["timestamp"]
      .diff()
      .value_counts()
      .head(10)
)

print("Most common time intervals per road:")
print(time_gaps)

Most common time intervals per road:
timestamp
0 days 00:30:00    172291
0 days 01:00:00      4272
0 days 01:30:00       113
Name: count, dtype: int64


In [11]:
# Sort by road and time
df = df.sort_values(
    ["road_id", "timestamp"]
).reset_index(drop=True)

# Prepare the next two observations for each road
grouped = df.groupby("road_id", sort=False)

next_time_1 = grouped["timestamp"].shift(-1)
next_time_2 = grouped["timestamp"].shift(-2)

next_accident_1 = grouped["accident_count"].shift(-1)
next_accident_2 = grouped["accident_count"].shift(-2)

# Confirm both future observations are exactly 30 minutes apart
valid_future = (
    ((next_time_1 - df["timestamp"]) == pd.Timedelta(minutes=30))
    & ((next_time_2 - next_time_1) == pd.Timedelta(minutes=30))
)

# New target: accident in either of the next two intervals
df["future_accident_risk_60m"] = pd.NA

df.loc[valid_future, "future_accident_risk_60m"] = (
    (next_accident_1[valid_future].fillna(0) > 0)
    | (next_accident_2[valid_future].fillna(0) > 0)
).astype(int)

print("Valid future windows:", int(valid_future.sum()))
print("\nFuture-risk target distribution:")
print(df["future_accident_risk_60m"].value_counts(dropna=False))

Valid future windows: 167990

Future-risk target distribution:
future_accident_risk_60m
0       164856
<NA>      8711
1         3134
Name: count, dtype: int64


C:\Users\pathi\AppData\Local\Temp\ipykernel_5864\131119916.py:17: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  ((next_time_1 - df["timestamp"]) == pd.Timedelta(minutes=30))
C:\Users\pathi\AppData\Local\Temp\ipykernel_5864\131119916.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  & ((next_time_2 - next_time_1) == pd.Timedelta(minutes=30))


In [12]:
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

df["timestamp"] = pd.to_datetime(
    df["timestamp"], errors="coerce"
)

df = df.dropna(subset=["timestamp"])

df = df.sort_values(
    "timestamp"
).reset_index(drop=True)
model_df = df.loc[
    df["future_accident_risk_60m"].notna()
].copy()

# Exclude targets, outcomes and identifiers from model inputs
exclude_cols = [
    "accident_risk",
    "accident_count",
    "future_accident_risk_60m",
    "timestamp",
    "date",
    "time",
    "road_id",
    "road_name",
    "weather_station_id",
    "weather_station"
]

feature_cols = [
    c for c in model_df.select_dtypes(include="number").columns
    if c not in exclude_cols
]

X = model_df[feature_cols].replace([np.inf, -np.inf], np.nan)
y = model_df["future_accident_risk_60m"].astype(int)

# Earlier records train the model; later records test it
split_idx = int(len(model_df) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Fit imputation using training data only
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(
    imputer.fit_transform(X_train), columns=feature_cols
)
X_test = pd.DataFrame(
    imputer.transform(X_test), columns=feature_cols
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training positive cases:", int(y_train.sum()))
print("Testing positive cases:", int(y_test.sum()))
print("Number of features:", len(feature_cols))
print("Features:", feature_cols)

Training rows: 134392
Testing rows: 33598
Training positive cases: 2484
Testing positive cases: 650
Number of features: 32
Features: ['latitude', 'longitude', 'traffic_volume', 'vehicle_count', 'avg_speed', 'occupancy', 'travel_time', 'signal_timing', 'road_capacity', 'capacity_30min', 'volume_capacity_ratio', 'temperature', 'rainfall', 'visibility', 'public_holiday', 'event_flag', 'roadwork_flag', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'is_morning_peak', 'is_evening_peak', 'traffic_volume_lag_1', 'traffic_volume_lag_2', 'traffic_volume_lag_4', 'avg_speed_lag_1', 'avg_speed_lag_2', 'traffic_volume_rolling_mean_4', 'avg_speed_rolling_mean_4']


In [16]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

for threshold in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    y_threshold_pred = (
        y_probability >= threshold
    ).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Precision:", round(
        precision_score(y_test, y_threshold_pred, zero_division=0), 4
    ))
    print("Recall:", round(
        recall_score(y_test, y_threshold_pred, zero_division=0), 4
    ))
    print("F1-score:", round(
        f1_score(y_test, y_threshold_pred, zero_division=0), 4
    ))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_threshold_pred))


Threshold: 0.05
Precision: 0.0193
Recall: 1.0
F1-score: 0.038
Confusion matrix:
[[    0 32948]
 [    0   650]]

Threshold: 0.1
Precision: 0.0194
Recall: 0.9985
F1-score: 0.038
Confusion matrix:
[[   71 32877]
 [    1   649]]

Threshold: 0.15
Precision: 0.0193
Recall: 0.9723
F1-score: 0.0378
Confusion matrix:
[[  774 32174]
 [   18   632]]

Threshold: 0.2
Precision: 0.0194
Recall: 0.8908
F1-score: 0.0379
Confusion matrix:
[[ 3659 29289]
 [   71   579]]

Threshold: 0.3
Precision: 0.022
Recall: 0.4246
F1-score: 0.0418
Confusion matrix:
[[20665 12283]
 [  374   276]]

Threshold: 0.5
Precision: 0.0402
Recall: 0.0815
F1-score: 0.0538
Confusion matrix:
[[31682  1266]
 [  597    53]]


In [14]:
print("Test labels:", len(y_test))
print("Predicted probabilities:", len(y_probability))
print("Test rows:", len(X_test))

Test labels: 33598
Predicted probabilities: 35341
Test rows: 33598


In [15]:
# Check the current sizes
print("y_test:", len(y_test))
print("X_test:", len(X_test))
print("y_probability:", len(y_probability))

# Recalculate probabilities using the current test data
y_probability = model.predict_proba(X_test)[:, 1]

print("\nAfter correction:")
print("y_test:", len(y_test))
print("X_test:", len(X_test))
print("y_probability:", len(y_probability))

y_test: 33598
X_test: 33598
y_probability: 35341

After correction:
y_test: 33598
X_test: 33598
y_probability: 33598


In [17]:
print("ROC-AUC:", round(roc_auc_score(y_test, y_probability), 4))
print("Positive cases:", int(y_test.sum()))
print("Total test rows:", len(y_test))
print("Positive rate (%):", round(y_test.mean() * 100, 2))

ROC-AUC: 0.5383
Positive cases: 650
Total test rows: 33598
Positive rate (%): 1.93


In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

improved_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

improved_model.fit(X_train, y_train)

improved_probability = improved_model.predict_proba(X_test)[:, 1]

print("Improved ROC-AUC:",
      round(roc_auc_score(y_test, improved_probability), 4))

print("Average Precision:",
      round(average_precision_score(y_test, improved_probability), 4))

Improved ROC-AUC: 0.5076
Average Precision: 0.0221


In [19]:
import pandas as pd

importance = pd.Series(
    improved_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("Top 15 features for accident-risk prediction:")
print(importance.head(15).round(4))

Top 15 features for accident-risk prediction:
volume_capacity_ratio            0.0549
travel_time                      0.0540
occupancy                        0.0538
avg_speed                        0.0534
temperature                      0.0525
traffic_volume_lag_4             0.0516
avg_speed_rolling_mean_4         0.0490
avg_speed_lag_2                  0.0479
traffic_volume_rolling_mean_4    0.0468
avg_speed_lag_1                  0.0463
vehicle_count                    0.0461
traffic_volume_lag_2             0.0456
traffic_volume_lag_1             0.0456
traffic_volume                   0.0450
day_of_month                     0.0430
dtype: float64


In [20]:
print("Features used by the model:")
print(X_train.columns.tolist())

print("\nTarget column:")
print("future_accident_risk_60m")

print("\nCurrent and future accident-related columns:")
print([
    col for col in df.columns
    if "accident" in col.lower() or "risk" in col.lower()
])

Features used by the model:
['latitude', 'longitude', 'traffic_volume', 'vehicle_count', 'avg_speed', 'occupancy', 'travel_time', 'signal_timing', 'road_capacity', 'capacity_30min', 'volume_capacity_ratio', 'temperature', 'rainfall', 'visibility', 'public_holiday', 'event_flag', 'roadwork_flag', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'is_morning_peak', 'is_evening_peak', 'traffic_volume_lag_1', 'traffic_volume_lag_2', 'traffic_volume_lag_4', 'avg_speed_lag_1', 'avg_speed_lag_2', 'traffic_volume_rolling_mean_4', 'avg_speed_rolling_mean_4']

Target column:
future_accident_risk_60m

Current and future accident-related columns:
['accident_count', 'accident_risk', 'future_accident_risk_60m']


In [21]:
# Check whether training and testing labels align with the features

print("Training features:", len(X_train))
print("Training labels:", len(y_train))

print("Testing features:", len(X_test))
print("Testing labels:", len(y_test))

print("\nMissing values in training labels:", y_train.isna().sum())
print("Missing values in testing labels:", y_test.isna().sum())

print("\nTarget distribution in test data:")
print(y_test.value_counts(dropna=False))

Training features: 134392
Training labels: 134392
Testing features: 33598
Testing labels: 33598

Missing values in training labels: 0
Missing values in testing labels: 0

Target distribution in test data:
future_accident_risk_60m
0    32948
1      650
Name: count, dtype: int64


In [22]:
print("Current accident-risk distribution:")
print(df["accident_risk"].value_counts(dropna=False))

print("\nFuture 60-minute accident-risk distribution:")
print(df["future_accident_risk_60m"].value_counts(dropna=False))

print("\nAccident counts:")
print(df["accident_count"].value_counts(dropna=False))

Current accident-risk distribution:
accident_risk
0    175049
1      1652
Name: count, dtype: int64

Future 60-minute accident-risk distribution:
future_accident_risk_60m
0       164856
<NA>      8711
1         3134
Name: count, dtype: int64

Accident counts:
accident_count
0    175049
1      1637
2        15
Name: count, dtype: int64


In [23]:
print("Missing future-risk labels by road:")
print(
    df.loc[df["future_accident_risk_60m"].isna()]
      .groupby("road_id")
      .size()
      .sort_values(ascending=False)
      .head(10)
)

print("\nTotal rows:", len(df))
print(
    "Missing future-risk labels:",
    df["future_accident_risk_60m"].isna().sum()
)

Missing future-risk labels by road:
road_id
NL-019    398
NL-015    396
NL-006    380
NL-022    376
NL-016    370
NL-024    367
NL-001    365
NL-005    361
NL-003    358
NL-018    358
dtype: int64

Total rows: 176701
Missing future-risk labels: 8711


In [24]:
# Inspect rows where the future accident-risk label is missing

missing_rows = df[df["future_accident_risk_60m"].isna()].copy()

print("First 15 rows with missing future-risk labels:")
print(
    missing_rows[
        ["road_id", "timestamp", "accident_count",
         "future_accident_risk_60m"]
    ].head(15).to_string(index=False)
)

print("\nTimestamp range of missing rows:")
print(missing_rows["timestamp"].min())
print(missing_rows["timestamp"].max())

First 15 rows with missing future-risk labels:
road_id           timestamp  accident_count future_accident_risk_60m
 NL-005 2025-01-01 00:00:00               0                     <NA>
 NL-018 2025-01-01 00:00:00               0                     <NA>
 NL-018 2025-01-01 00:30:00               0                     <NA>
 NL-005 2025-01-01 00:30:00               0                     <NA>
 NL-021 2025-01-01 00:30:00               0                     <NA>
 NL-021 2025-01-01 01:00:00               0                     <NA>
 NL-025 2025-01-01 01:00:00               0                     <NA>
 NL-025 2025-01-01 01:30:00               0                     <NA>
 NL-006 2025-01-01 02:00:00               0                     <NA>
 NL-021 2025-01-01 02:00:00               0                     <NA>
 NL-006 2025-01-01 02:30:00               0                     <NA>
 NL-001 2025-01-01 03:00:00               0                     <NA>
 NL-001 2025-01-01 03:30:00               0             

In [25]:
# Check time gaps for rows with missing future-risk labels

df = df.sort_values(["road_id", "timestamp"]).copy()

df["next_timestamp"] = (
    df.groupby("road_id")["timestamp"].shift(-1)
)

missing_rows = df[df["future_accident_risk_60m"].isna()]

print("Next timestamp for missing-label rows:")
print(
    missing_rows[
        ["road_id", "timestamp", "next_timestamp"]
    ].head(15).to_string(index=False)
)

print("\nTime difference to next observation:")
print(
    (
        missing_rows["next_timestamp"]
        - missing_rows["timestamp"]
    ).value_counts(dropna=False).head(10)
)

Next timestamp for missing-label rows:
road_id           timestamp      next_timestamp
 NL-001 2025-01-01 03:00:00 2025-01-01 03:30:00
 NL-001 2025-01-01 03:30:00 2025-01-01 04:30:00
 NL-001 2025-01-02 01:00:00 2025-01-02 01:30:00
 NL-001 2025-01-02 01:30:00 2025-01-02 02:30:00
 NL-001 2025-01-02 02:30:00 2025-01-02 03:00:00
 NL-001 2025-01-02 03:00:00 2025-01-02 04:00:00
 NL-001 2025-01-03 08:00:00 2025-01-03 08:30:00
 NL-001 2025-01-03 08:30:00 2025-01-03 09:30:00
 NL-001 2025-01-04 00:30:00 2025-01-04 01:00:00
 NL-001 2025-01-04 01:00:00 2025-01-04 02:00:00
 NL-001 2025-01-04 23:00:00 2025-01-04 23:30:00
 NL-001 2025-01-04 23:30:00 2025-01-05 00:30:00
 NL-001 2025-01-05 15:30:00 2025-01-05 16:00:00
 NL-001 2025-01-05 16:00:00 2025-01-05 17:00:00
 NL-001 2025-01-05 20:00:00 2025-01-05 20:30:00

Time difference to next observation:
0 days 00:30:00    4301
0 days 01:00:00    4272
0 days 01:30:00     113
NaT                  25
Name: count, dtype: int64


In [26]:
# Check the next two timestamps for each road

df = df.sort_values(["road_id", "timestamp"]).copy()

grouped = df.groupby("road_id")

df["next_time_1"] = grouped["timestamp"].shift(-1)
df["next_time_2"] = grouped["timestamp"].shift(-2)

valid_30min = (
    (df["next_time_1"] - df["timestamp"] == pd.Timedelta(minutes=30))
    & (df["next_time_2"] - df["next_time_1"] == pd.Timedelta(minutes=30))
)

print("Rows with two valid future intervals:", int(valid_30min.sum()))
print("Rows without two valid future intervals:", int((~valid_30min).sum()))

print("\nMissing labels with valid future intervals:")
print(
    int(
        (
            df["future_accident_risk_60m"].isna()
            & valid_30min
        ).sum()
    )
)

Rows with two valid future intervals: 167990
Rows without two valid future intervals: 8711

Missing labels with valid future intervals:
0


C:\Users\pathi\AppData\Local\Temp\ipykernel_5864\1785245976.py:11: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  (df["next_time_1"] - df["timestamp"] == pd.Timedelta(minutes=30))
C:\Users\pathi\AppData\Local\Temp\ipykernel_5864\1785245976.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  & (df["next_time_2"] - df["next_time_1"] == pd.Timedelta(minutes=30))


In [27]:
# Check how the 60-minute future-risk target was created

print("Current accident-risk column:")
print(df["accident_risk"].value_counts(dropna=False))

print("\nFuture accident-risk target:")
print(df["future_accident_risk_60m"].value_counts(dropna=False))

print("\nRows with missing future-risk labels:")
print(df["future_accident_risk_60m"].isna().sum())

print("\nCheck whether missing labels have valid future intervals:")
print(
    df.loc[
        df["future_accident_risk_60m"].isna(),
        ["road_id", "timestamp", "accident_count",
         "next_time_1", "next_time_2"]
    ].head(10).to_string(index=False)
)

Current accident-risk column:
accident_risk
0    175049
1      1652
Name: count, dtype: int64

Future accident-risk target:
future_accident_risk_60m
0       164856
<NA>      8711
1         3134
Name: count, dtype: int64

Rows with missing future-risk labels:
8711

Check whether missing labels have valid future intervals:
road_id           timestamp  accident_count         next_time_1         next_time_2
 NL-001 2025-01-01 03:00:00               0 2025-01-01 03:30:00 2025-01-01 04:30:00
 NL-001 2025-01-01 03:30:00               0 2025-01-01 04:30:00 2025-01-01 05:00:00
 NL-001 2025-01-02 01:00:00               0 2025-01-02 01:30:00 2025-01-02 02:30:00
 NL-001 2025-01-02 01:30:00               0 2025-01-02 02:30:00 2025-01-02 03:00:00
 NL-001 2025-01-02 02:30:00               0 2025-01-02 03:00:00 2025-01-02 04:00:00
 NL-001 2025-01-02 03:00:00               0 2025-01-02 04:00:00 2025-01-02 04:30:00
 NL-001 2025-01-03 08:00:00               0 2025-01-03 08:30:00 2025-01-03 09:30:00
 NL-0

In [28]:
# Check whether the next observations are exactly 30 and 60 minutes later

check = df[
    ["road_id", "timestamp", "next_time_1", "next_time_2",
     "future_accident_risk_60m"]
].copy()

check["gap_1_minutes"] = (
    check["next_time_1"] - check["timestamp"]
).dt.total_seconds() / 60

check["gap_2_minutes"] = (
    check["next_time_2"] - check["timestamp"]
).dt.total_seconds() / 60

print("Time gap to next observation:")
print(check["gap_1_minutes"].value_counts(dropna=False).head(10))

print("\nTime gap to second future observation:")
print(check["gap_2_minutes"].value_counts(dropna=False).head(10))

print("\nMissing labels with their time gaps:")
print(
    check.loc[
        check["future_accident_risk_60m"].isna(),
        ["timestamp", "next_time_1", "next_time_2",
         "gap_1_minutes", "gap_2_minutes"]
    ].head(10).to_string(index=False)
)

Time gap to next observation:
gap_1_minutes
30.0    172291
60.0      4272
90.0       113
NaN         25
Name: count, dtype: int64

Time gap to second future observation:
gap_2_minutes
60.0     167990
90.0       8337
120.0       314
NaN          50
150.0        10
Name: count, dtype: int64

Missing labels with their time gaps:
          timestamp         next_time_1         next_time_2  gap_1_minutes  gap_2_minutes
2025-01-01 03:00:00 2025-01-01 03:30:00 2025-01-01 04:30:00           30.0           90.0
2025-01-01 03:30:00 2025-01-01 04:30:00 2025-01-01 05:00:00           60.0           90.0
2025-01-02 01:00:00 2025-01-02 01:30:00 2025-01-02 02:30:00           30.0           90.0
2025-01-02 01:30:00 2025-01-02 02:30:00 2025-01-02 03:00:00           60.0           90.0
2025-01-02 02:30:00 2025-01-02 03:00:00 2025-01-02 04:00:00           30.0           90.0
2025-01-02 03:00:00 2025-01-02 04:00:00 2025-01-02 04:30:00           60.0           90.0
2025-01-03 08:00:00 2025-01-03 08:30:00 20

In [29]:
# Evaluate the accident-risk prediction model

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining the Random Forest model...")

model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

model.fit(X_train_imputed, y_train)

y_pred = model.predict(X_test_imputed)
y_prob = model.predict_proba(X_test_imputed)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("Average Precision:", average_precision_score(y_test, y_prob))

Training rows: 134392
Testing rows: 33598

Training the Random Forest model...

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     32948
           1       0.00      0.00      0.00       650

    accuracy                           0.98     33598
   macro avg       0.49      0.50      0.50     33598
weighted avg       0.96      0.98      0.97     33598

ROC-AUC: 0.5372043126231544
Average Precision: 0.02167210992798108


In [30]:
# Check accident-risk class balance and feature importance

print("Training target distribution:")
print(y_train.value_counts(normalize=True).rename("proportion"))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).rename("proportion"))

importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("\nTop 15 feature importances:")
print(importance.head(15))

Training target distribution:
future_accident_risk_60m
0    0.981517
1    0.018483
Name: proportion, dtype: float64

Testing target distribution:
future_accident_risk_60m
0    0.980654
1    0.019346
Name: proportion, dtype: float64

Top 15 feature importances:
temperature                      0.055344
avg_speed                        0.052934
travel_time                      0.052787
occupancy                        0.052714
volume_capacity_ratio            0.052606
traffic_volume_lag_4             0.052312
avg_speed_rolling_mean_4         0.050214
avg_speed_lag_1                  0.049767
avg_speed_lag_2                  0.049632
traffic_volume_lag_1             0.048632
traffic_volume_lag_2             0.048229
traffic_volume_rolling_mean_4    0.047653
day_of_month                     0.046386
vehicle_count                    0.046303
traffic_volume                   0.043475
dtype: float64


In [31]:
# Improve accident-risk detection with class weighting

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

balanced_model = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced_subsample",
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

balanced_model.fit(X_train_imputed, y_train)

risk_probability = balanced_model.predict_proba(X_test_imputed)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, risk_probability))
print("Average Precision:", average_precision_score(y_test, risk_probability))

# Use a lower threshold to help identify rare risk cases
risk_prediction = (risk_probability >= 0.20).astype(int)

print("\nClassification Report at threshold 0.20:")
print(classification_report(
    y_test,
    risk_prediction,
    zero_division=0
))

ROC-AUC: 0.5554812244936077
Average Precision: 0.024799935748292895

Classification Report at threshold 0.20:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     32948
           1       0.00      0.00      0.00       650

    accuracy                           0.98     33598
   macro avg       0.49      0.50      0.49     33598
weighted avg       0.96      0.98      0.97     33598



In [32]:
 # Check whether the current train/test split is time-based

print("Training timestamp range:")
print(df.loc[X_train.index, "timestamp"].min())
print(df.loc[X_train.index, "timestamp"].max())

print("\nTesting timestamp range:")
print(df.loc[X_test.index, "timestamp"].min())
print(df.loc[X_test.index, "timestamp"].max())

Training timestamp range:
2025-01-01 00:00:00
2025-04-25 19:00:00

Testing timestamp range:
2025-01-01 00:00:00
2025-01-29 17:00:00


In [33]:
# Check whether training and testing data overlap in time
print("Training data starts:", df.loc[X_train.index, "timestamp"].min())
print("Training data ends:", df.loc[X_train.index, "timestamp"].max())

print("Testing data starts:", df.loc[X_test.index, "timestamp"].min())
print("Testing data ends:", df.loc[X_test.index, "timestamp"].max())

Training data starts: 2025-01-01 00:00:00
Training data ends: 2025-04-25 19:00:00
Testing data starts: 2025-01-01 00:00:00
Testing data ends: 2025-01-29 17:00:00


In [34]:
# Create a chronological 80/20 train-test split

df = df.sort_values("timestamp").copy()

# Keep rows that have a valid future accident-risk label
model_df = df.dropna(subset=["future_accident_risk_60m"]).copy()

# Split at the 80% time point
split_position = int(len(model_df) * 0.80)
split_time = model_df["timestamp"].iloc[split_position]

train_df = model_df[model_df["timestamp"] < split_time].copy()
test_df = model_df[model_df["timestamp"] >= split_time].copy()

print("Split timestamp:", split_time)
print("Training dates:", train_df["timestamp"].min(), "to", train_df["timestamp"].max())
print("Testing dates:", test_df["timestamp"].min(), "to", test_df["timestamp"].max())
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

Split timestamp: 2025-05-01 17:00:00
Training dates: 2025-01-01 00:00:00 to 2025-05-01 16:30:00
Testing dates: 2025-05-01 17:00:00 to 2025-05-31 22:30:00
Training rows: 134373
Testing rows: 33617


In [35]:
# Rebuild features and target using the chronological split

exclude_cols = [
    "accident_risk",
    "accident_count",
    "future_accident_risk_60m",
    "timestamp",
    "date",
    "time",
    "road_id",
    "road_name",
    "weather_station_id",
    "weather_station"
]

feature_cols = [
    col for col in model_df.select_dtypes(include="number").columns
    if col not in exclude_cols
]

X_train = train_df[feature_cols].copy()
y_train = train_df["future_accident_risk_60m"].astype(int)

X_test = test_df[feature_cols].copy()
y_test = test_df["future_accident_risk_60m"].astype(int)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts())
print("\nTesting target distribution:")
print(y_test.value_counts())

Training features: (134373, 32)
Testing features: (33617, 32)

Training target distribution:
future_accident_risk_60m
0    131889
1      2484
Name: count, dtype: int64

Testing target distribution:
future_accident_risk_60m
0    32967
1      650
Name: count, dtype: int64


In [36]:
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)

# Handle missing feature values using training data only
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Train a class-balanced Random Forest
model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training model...")
model.fit(X_train_imputed, y_train)

# Predict accident-risk probabilities
y_prob = model.predict_proba(X_test_imputed)[:, 1]

# Start with a threshold of 0.20
y_pred = (y_prob >= 0.20).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("Average Precision:", round(average_precision_score(y_test, y_prob), 4))

Training model...

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     32967
           1       0.00      0.00      0.00       650

    accuracy                           0.98     33617
   macro avg       0.49      0.50      0.50     33617
weighted avg       0.96      0.98      0.97     33617

ROC-AUC: 0.5379
Average Precision: 0.0224


In [37]:
 import numpy as np

print("Actual positive cases:", int(y_test.sum()))
print("Predicted positive cases at threshold 0.20:",
      int((y_prob >= 0.20).sum()))

print("\nProbability distribution:")
print("Minimum:", round(float(np.min(y_prob)), 4))
print("25th percentile:", round(float(np.percentile(y_prob, 25)), 4))
print("Median:", round(float(np.median(y_prob)), 4))
print("90th percentile:", round(float(np.percentile(y_prob, 90)), 4))
print("99th percentile:", round(float(np.percentile(y_prob, 99)), 4))
print("Maximum:", round(float(np.max(y_prob)), 4))

Actual positive cases: 650
Predicted positive cases at threshold 0.20: 7

Probability distribution:
Minimum: 0.0
25th percentile: 0.0
Median: 0.01
90th percentile: 0.04
99th percentile: 0.09
Maximum: 0.28


In [38]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Threshold | Predicted Risk | Precision | Recall | F1-score")
print("-" * 65)

for threshold in [0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]:
    predictions = (y_prob >= threshold).astype(int)

    print(
        f"{threshold:8.2f} | "
        f"{predictions.sum():14d} | "
        f"{precision_score(y_test, predictions, zero_division=0):9.3f} | "
        f"{recall_score(y_test, predictions, zero_division=0):6.3f} | "
        f"{f1_score(y_test, predictions, zero_division=0):8.3f}"
    )

Threshold | Predicted Risk | Precision | Recall | F1-score
-----------------------------------------------------------------
    0.01 |          24199 |     0.020 |  0.751 |    0.039
    0.02 |          14826 |     0.022 |  0.498 |    0.042
    0.03 |           8514 |     0.023 |  0.303 |    0.043
    0.05 |           2794 |     0.029 |  0.123 |    0.046
    0.08 |            671 |     0.030 |  0.031 |    0.030
    0.10 |            281 |     0.032 |  0.014 |    0.019
    0.15 |             41 |     0.024 |  0.002 |    0.003
    0.20 |              7 |     0.000 |  0.000 |    0.000


In [39]:
# Check the strongest features used by the accident-risk model

importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("Top 15 features:")
print(importance.head(15).to_string())

print("\nAccident count distribution:")
print(df["accident_count"].value_counts().head(10))

print("\nCurrent accident-risk distribution:")
print(df["accident_risk"].value_counts(dropna=False))

Top 15 features:
temperature                      0.055212
avg_speed                        0.054394
travel_time                      0.053280
occupancy                        0.053186
volume_capacity_ratio            0.052627
traffic_volume_lag_4             0.052021
avg_speed_rolling_mean_4         0.050755
avg_speed_lag_1                  0.050022
avg_speed_lag_2                  0.049452
traffic_volume_lag_2             0.047956
traffic_volume_rolling_mean_4    0.047565
traffic_volume_lag_1             0.047107
day_of_month                     0.046431
traffic_volume                   0.044848
vehicle_count                    0.044432

Accident count distribution:
accident_count
0    175049
1      1637
2        15
Name: count, dtype: int64

Current accident-risk distribution:
accident_risk
0    175049
1      1652
Name: count, dtype: int64


In [40]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)

# Give more importance to rare accident cases
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

balanced_model = RandomForestClassifier(
    n_estimators=150,
    class_weight={0: 1, 1: 20},
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

print("Training improved accident-risk model...")
balanced_model.fit(X_train_imputed, y_train)

risk_prob = balanced_model.predict_proba(X_test_imputed)[:, 1]

print("\nROC-AUC:", round(roc_auc_score(y_test, risk_prob), 4))
print("Average Precision:", round(average_precision_score(y_test, risk_prob), 4))

risk_pred = (risk_prob >= 0.10).astype(int)

print("\nClassification report at threshold 0.10:")
print(classification_report(y_test, risk_pred, zero_division=0))
print("Predicted accident-risk cases:", int(risk_pred.sum()))
print("Actual accident-risk cases:", int(y_test.sum()))

Training improved accident-risk model...

ROC-AUC: 0.5664
Average Precision: 0.0264

Classification report at threshold 0.10:
              precision    recall  f1-score   support

           0       0.98      0.93      0.96     32967
           1       0.03      0.11      0.05       650

    accuracy                           0.92     33617
   macro avg       0.51      0.52      0.50     33617
weighted avg       0.96      0.92      0.94     33617

Predicted accident-risk cases: 2245
Actual accident-risk cases: 650


In [41]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    {
        "model": balanced_model,
        "imputer": imputer,
        "features": list(X_train.columns),
        "threshold": 0.10
    },
    "../models/accident_risk_60m_model.joblib"
)

print("60-minute accident-risk model saved successfully.")
print("Location: models/accident_risk_60m_model.joblib")

60-minute accident-risk model saved successfully.
Location: models/accident_risk_60m_model.joblib


In [42]:
import joblib
import os

model_path = "../models/accident_risk_60m_model.joblib"

saved_data = joblib.load(model_path)

print("Model file exists:", os.path.exists(model_path))
print("Model loaded:", saved_data["model"] is not None)
print("Number of features:", len(saved_data["features"]))
print("Saved threshold:", saved_data["threshold"])
print("Model verification completed.")

Model file exists: True
Model loaded: True
Number of features: 32
Saved threshold: 0.1
Model verification completed.


In [43]:
from pathlib import Path

model_path = Path("../models/accident_risk_60m_model.joblib")

print("Current folder:", Path.cwd())
print("Model file exists:", model_path.exists())
print("Model path:", model_path.resolve())

Current folder: C:\Users\pathi\Desktop\FlowCast\notebooks
Model file exists: True
Model path: C:\Users\pathi\Desktop\FlowCast\models\accident_risk_60m_model.joblib


In [44]:
import joblib

model = joblib.load(model_path)

print("Model loaded successfully!")
print("Model type:", type(model))

Model loaded successfully!
Model type: <class 'dict'>


In [45]:
print("Model keys:", model.keys())

for key, value in model.items():
    print(f"\nKey: {key}")
    print("Type:", type(value))

Model keys: dict_keys(['model', 'imputer', 'features', 'threshold'])

Key: model
Type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>

Key: imputer
Type: <class 'sklearn.impute._base.SimpleImputer'>

Key: features
Type: <class 'list'>

Key: threshold
Type: <class 'float'>


In [46]:
print("Feature information:")
print(model["features"])

print("\nDecision threshold:")
print(model["threshold"])

print("\nClassifier:")
print(model["model"])

Feature information:
['latitude', 'longitude', 'traffic_volume', 'vehicle_count', 'avg_speed', 'occupancy', 'travel_time', 'signal_timing', 'road_capacity', 'capacity_30min', 'volume_capacity_ratio', 'temperature', 'rainfall', 'visibility', 'public_holiday', 'event_flag', 'roadwork_flag', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'is_morning_peak', 'is_evening_peak', 'traffic_volume_lag_1', 'traffic_volume_lag_2', 'traffic_volume_lag_4', 'avg_speed_lag_1', 'avg_speed_lag_2', 'traffic_volume_rolling_mean_4', 'avg_speed_rolling_mean_4']

Decision threshold:
0.1

Classifier:
RandomForestClassifier(class_weight={0: 1, 1: 20}, min_samples_leaf=3,
                       n_estimators=150, n_jobs=-1, random_state=42)


In [47]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

extra_model = ExtraTreesClassifier(
    n_estimators=200,
    max_features="sqrt",
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

extra_model.fit(X_train_imputed, y_train)

extra_prob = extra_model.predict_proba(X_test_imputed)[:, 1]

print("Extra Trees ROC-AUC:", round(roc_auc_score(y_test, extra_prob), 4))
print("Average Precision:", round(average_precision_score(y_test, extra_prob), 4))

Extra Trees ROC-AUC: 0.5779
Average Precision: 0.0261


In [48]:
import joblib
import os

extra_bundle = {
    "model": extra_model,
    "imputer": imputer,
    "features": list(X_train.columns),
    "threshold": 0.10
}

os.makedirs("../models", exist_ok=True)

joblib.dump(
    extra_bundle,
    "../models/accident_risk_60m_extra_trees.joblib"
)

print("Extra Trees model saved successfully!")

Extra Trees model saved successfully!


In [51]:
import os
import joblib

model_path = "../models/accident_risk_60m_extra_trees.joblib"

print("File exists:", os.path.exists(model_path))

loaded_bundle = joblib.load(model_path)

print("Model:", type(loaded_bundle["model"]).__name__)
print("Number of features:", len(loaded_bundle["features"]))
print("Threshold:", loaded_bundle["threshold"])

File exists: True
Model: ExtraTreesClassifier
Number of features: 32
Threshold: 0.1
